# Signal, Shared Noise, and Local Noise

Notebook 00 demonstrated that differential measurements can cancel common-mode noise.

This notebook asks a deeper question:

**What ultimately limits differential sensing?**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("..")
FIGURES = ROOT / "figures"
DATA = ROOT / "data"

FIGURES.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

rng = np.random.default_rng(42)



## Differential sensing model

\[
\phi_A = +s/2 + n_c + n_A
\]

\[
\phi_B = -s/2 + n_c + n_B
\]

\[
\phi_{diff} = \phi_A - \phi_B
\]

The shared contribution cancels.


In [ ]:
n = 1000
t = np.linspace(0, 10, n)
signal = 0.3 * np.sin(2 * np.pi * 0.5 * t)


## Recovery versus common-mode noise

In [ ]:
common_levels = np.linspace(0, 5, 25)

rmse_common = []

for amp in common_levels:
    common = amp * rng.normal(size=n)

    local_a = 0.15 * rng.normal(size=n)
    local_b = 0.15 * rng.normal(size=n)

    phi_a = signal / 2 + common + local_a
    phi_b = -signal / 2 + common + local_b

    phi_diff = phi_a - phi_b

    rmse = np.sqrt(np.mean((phi_diff - signal) ** 2))
    rmse_common.append(rmse)


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(common_levels, rmse_common, marker="o")
plt.xlabel("Common noise amplitude")
plt.ylabel("Differential RMSE")
plt.title("Differential recovery is insensitive to common-mode noise")
plt.tight_layout()
plt.savefig(FIGURES / "07_common_noise_sweep.png", dpi=200)
plt.show()


## Recovery versus local noise

In [ ]:
local_levels = np.linspace(0, 2, 25)

rmse_local = []

for amp in local_levels:
    common = 2.0 * rng.normal(size=n)

    local_a = amp * rng.normal(size=n)
    local_b = amp * rng.normal(size=n)

    phi_a = signal / 2 + common + local_a
    phi_b = -signal / 2 + common + local_b

    phi_diff = phi_a - phi_b

    rmse = np.sqrt(np.mean((phi_diff - signal) ** 2))
    rmse_local.append(rmse)


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(local_levels, rmse_local, marker="o")
plt.xlabel("Local noise amplitude")
plt.ylabel("Differential RMSE")
plt.title("Local noise limits differential sensing")
plt.tight_layout()
plt.savefig(FIGURES / "07_local_noise_sweep.png", dpi=200)
plt.show()


## Signal recovery map

In [ ]:
common_grid = np.linspace(0, 5, 40)
local_grid = np.linspace(0, 2, 40)

heatmap = np.zeros((40, 40))

for i, common_amp in enumerate(common_grid):
    for j, local_amp in enumerate(local_grid):

        common = common_amp * rng.normal(size=n)

        local_a = local_amp * rng.normal(size=n)
        local_b = local_amp * rng.normal(size=n)

        phi_a = signal / 2 + common + local_a
        phi_b = -signal / 2 + common + local_b

        phi_diff = phi_a - phi_b

        heatmap[j, i] = np.sqrt(
            np.mean((phi_diff - signal) ** 2)
        )


In [ ]:
plt.figure(figsize=(8,5))
plt.imshow(
    heatmap,
    origin="lower",
    aspect="auto",
    extent=[
        common_grid.min(),
        common_grid.max(),
        local_grid.min(),
        local_grid.max()
    ]
)

plt.colorbar(label="RMSE")
plt.xlabel("Common noise amplitude")
plt.ylabel("Local noise amplitude")
plt.title("Recovery depends primarily on local noise")

plt.tight_layout()
plt.savefig(FIGURES / "07_signal_recovery_map.png", dpi=200)
plt.show()


In [ ]:
summary = pd.DataFrame({
    "finding": [
        "common noise",
        "local noise",
        "differential sensing"
    ],
    "effect": [
        "cancels",
        "accumulates",
        "preserves signal"
    ]
})

summary.to_csv(DATA / "07_signal_noise_summary.csv", index=False)
summary


# Lesson

Common-mode noise is not the primary limitation of differential sensing.

Shared disturbances can be rejected because they affect both sensors similarly.

Local uncorrelated noise remains after subtraction and therefore sets the ultimate precision limit.

Notebook 13 will introduce correlation structure directly and show how recovery degrades as noise becomes less shared.
